Before you turn this problem in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel$\rightarrow$Restart) and then **run all cells** (in the menubar, select Cell$\rightarrow$Run All).

Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE", as well as your name and collaborators below:

In [ ]:
NAME = ""
COLLABORATORS = ""

---

# Lab 6 Backtrader Fundamental sol

In [ ]:
import backtrader as bt
from datetime import datetime
import pandas as pd

# Load data from CSV
df = pd.read_csv('MSFT.csv', index_col='Date', parse_dates=True)
df = df[df.index >= '2025-01-01']
df = df[df.index < '2025-10-01']
pandasdata = bt.feeds.PandasData(dataname=df, open=0, high=1, low=2, close=3, volume=5)

### Backtest a strategy with backtrader that buys 100 shares of MSFT at the first day start_date. Hold the 100 shares until end_date without buying additional shares. Output the return of this strategy.

In [ ]:
class TestStrategy(bt.Strategy):
    # Put your strategy below.
    def log(self, txt, dt=None):
        ''' Logging function for this strategy'''
        dt = dt or self.datas[0].datetime.date(0)
        print('%s, %s' % (dt.isoformat(), txt))

    def __init__(self):
        # Keep a reference to the "close" line in the data[0] dataseries
        self.dataclose = self.data.close

    def next(self):
        if (len(self)==1):
            self.buy(size=100)
            
    
    
    
        

In [ ]:
if __name__ == '__main__':
    cerebro = bt.Cerebro()
    init_cash = 100000.0
    cerebro.broker.setcash(init_cash)
    cerebro.addstrategy(TestStrategy)
    cerebro.adddata(pandasdata, name='MSFT')
    cerebro.run()
    value = cerebro.broker.get_value()
    cash_balance = cerebro.broker.get_cash()
    position = cerebro.broker.getposition(pandasdata)
    ret = (value-init_cash)/init_cash
    print (f'The one year return of the strategy is: {ret}')
    print (f'The cash left is: {cash_balance}')
    print (f'Current position: {position.size} shares of MSFT')


### Backtest a strategy with backtrader that ALL-IN MSFT from the first day start_date. Hold all the shares until end_date. Output the return of this strategy.

In [ ]:
class TestStrategy(bt.Strategy):
    # Put your strategy below.

    def log(self, txt, dt=None):
        ''' Logging function for this strategy'''
        dt = dt or self.datas[0].datetime.date(0)
        print('%s, %s' % (dt.isoformat(), txt))

    def __init__(self):
        # Keep a reference to the "close" line in the data[0] dataseries
        self.dataclose = self.data.close
        

    def next(self):
        # The self.order_target_percent() method in Backtrader is a convenient way to execute 
        # trades based on a target percentage of the available cash.
        self.order_target_percent(target=1.0)

In [ ]:
if __name__ == '__main__':
    cerebro = bt.Cerebro()
    init_cash = 100000.0
    cerebro.broker.setcash(init_cash)
    cerebro.addstrategy(TestStrategy)
    cerebro.adddata(pandasdata, name='MSFT')
    cerebro.run()
    value = cerebro.broker.get_value()
    cash_balance = cerebro.broker.get_cash()
    position = cerebro.broker.getposition(pandasdata)
    ret = (value-init_cash)/init_cash
    print (f'The one year return of the strategy is: {ret}')
    print (f'The cash left is: {cash_balance}')
    print (f'Current position: {position.size} shares of MSFT')


### Backtest a strategy with backtrader that buys 100 shares of MSFT every 30 trading days until out of cash from the first day start_date. For example, the first buying date is the 1st trading day since start_date, the second buying date is the 31th trading day, the third is 61 trading date, etc. Hold all the shares until end_date. Output the return of this strategy.

In [ ]:
class TestStrategy(bt.Strategy):
    # Put your strategy below.
 
    def log(self, txt, dt=None):
        ''' Logging function for this strategy'''
        dt = dt or self.datas[0].datetime.date(0)
        print('%s, %s' % (dt.isoformat(), txt))

    def __init__(self):
        # Keep a reference to the "close" line in the data[0] dataseries
        self.dataclose = self.data.close
        
    def notify_order(self, order):
        if order.status in [order.Completed]:
            if order.isbuy():
                # Buy order was filled
                filled_price = order.executed.price
                self.log(f"Buy order filled at price: {filled_price}")

    def next(self):
        if (len(self)%30==1):
            self.buy(size=100)
        

In [ ]:
if __name__ == '__main__':
    cerebro = bt.Cerebro()
    init_cash = 100000.0
    cerebro.broker.setcash(init_cash)
    cerebro.addstrategy(TestStrategy)
    cerebro.adddata(pandasdata, name='MSFT')
    cerebro.run()
    value = cerebro.broker.get_value()
    cash_balance = cerebro.broker.get_cash()
    position = cerebro.broker.getposition(pandasdata)
    ret = (value-init_cash)/init_cash
    print (f'The one year return of the strategy is: {ret}')
    print (f'The cash left is: {cash_balance}')
    print (f'Current position: {position.size} shares of MSFT')


### Backtest a strategy with backtrader that buys 100 shares of MSFT when the loss of a day is over 2\% (compare the close price with open price in one day)

In [ ]:
class TestStrategy(bt.Strategy):
    # Put your strategy below.
    
    def log(self, txt, dt=None):
        ''' Logging function for this strategy'''
        dt = dt or self.datas[0].datetime.date(0)
        print('%s, %s' % (dt.isoformat(), txt))

        
    def notify_order(self, order):
        if order.status in [order.Completed]:
            if order.isbuy():
                # Buy order was filled
                filled_price = order.executed.price
                self.log(f"Buy order filled at price: {filled_price}")

    def next(self):
        close_price,open_price = self.data.close[0], self.data.open[0]
        ret = (close_price-open_price)/open_price
        if (ret < -0.02):
            self.buy(size=100)    
    

In [ ]:
if __name__ == '__main__':
    cerebro = bt.Cerebro()
    init_cash = 100000.0
    cerebro.broker.setcash(init_cash)
    cerebro.addstrategy(TestStrategy)
    cerebro.adddata(pandasdata, name='MSFT')
    cerebro.run()
    value = cerebro.broker.get_value()
    cash_balance = cerebro.broker.get_cash()
    position = cerebro.broker.getposition(pandasdata)
    ret = (value-init_cash)/init_cash
    print (f'The one year return of the strategy is: {ret}')
    print (f'The cash left is: {cash_balance}')
    print (f'Current position: {position.size} shares of MSFT')
